In [1]:
# ================================================================
# ALL-SCHEMA-TYPES.CSV
# NLP + K-MEANS CLUSTERING
# Complete Jupyter Notebook
# ================================================================


# ================================================================
# CELL 1 — INSTALL REQUIRED LIBRARIES
# ================================================================

# Run this cell once if the libraries are not installed.

# !pip install pandas numpy scikit-learn matplotlib seaborn openpyxl


# ================================================================
# CELL 2 — IMPORT LIBRARIES
# ================================================================

import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ================================================================
# CELL 3 — CONFIGURATION
# ================================================================

FILE_PATH = "all-schema-types.csv"

OUTPUT_CLUSTERED = "all-schema-types_clustered.csv"
OUTPUT_SUMMARY = "all-schema-types_cluster_summary.csv"
OUTPUT_TERMS = "all-schema-types_cluster_terms.csv"

RANDOM_STATE = 42

# Number of clusters to test
MIN_K = 2
MAX_K = 10

# Maximum number of TF-IDF features
MAX_TFIDF_FEATURES = 5000

# Number of SVD components
SVD_COMPONENTS = 100

# Number of important words per cluster
TOP_TERMS = 20

print("Configuration ready.")


# ================================================================
# CELL 4 — CHECK THAT CSV EXISTS
# ================================================================

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"Could not find '{FILE_PATH}'. "
        "Place the CSV file in the same directory as this notebook."
    )

print(f"Found dataset: {FILE_PATH}")


# ================================================================
# CELL 5 — LOAD DATASET
# ================================================================

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print()
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


# ================================================================
# CELL 6 — DATASET INFORMATION
# ================================================================

print("DATASET INFORMATION")
print("=" * 70)

print("\nShape:")
print(df.shape)

print("\nColumn names:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_values")
)

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ================================================================
# CELL 7 — REMOVE DUPLICATES
# ================================================================

original_rows = len(df)

df = df.drop_duplicates().reset_index(drop=True)

removed_duplicates = original_rows - len(df)

print(f"Original rows: {original_rows}")
print(f"Duplicates removed: {removed_duplicates}")
print(f"Remaining rows: {len(df)}")


# ================================================================
# CELL 8 — IDENTIFY COLUMN TYPES
# ================================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nNUMERIC COLUMNS")
print("=" * 70)

if numeric_columns:
    for column in numeric_columns:
        print(column)
else:
    print("No numeric columns found.")


print("\nTEXT COLUMNS")
print("=" * 70)

if text_columns:
    for column in text_columns:
        print(column)
else:
    print("No text columns found.")


# ================================================================
# CELL 9 — CLEAN NUMERIC COLUMNS
# ================================================================

if numeric_columns:

    numeric_data = df[numeric_columns].copy()

    # Replace infinity values
    numeric_data = numeric_data.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Convert missing numeric values to median
    for column in numeric_columns:

        median_value = numeric_data[column].median()

        if pd.isna(median_value):
            median_value = 0

        numeric_data[column] = (
            numeric_data[column]
            .fillna(median_value)
        )

    print("Numeric data cleaned.")

else:

    numeric_data = pd.DataFrame(
        index=df.index
    )

    print("No numeric features to process.")


# ================================================================
# CELL 10 — CREATE COMBINED NLP TEXT
# ================================================================

if text_columns:

    df["combined_text"] = (
        df[text_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

else:

    # If there are no text columns, create an empty column
    df["combined_text"] = ""


print("Combined text created.")

display(
    df[["combined_text"]].head(10)
)


# ================================================================
# CELL 11 — TEXT CLEANING FUNCTION
# ================================================================

def clean_text(text):

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Keep letters and numbers
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["clean_text"] = (
    df["combined_text"]
    .apply(clean_text)
)

print("Text cleaning completed.")

display(
    df[
        ["combined_text", "clean_text"]
    ].head(10)
)


# ================================================================
# CELL 12 — CHECK EMPTY TEXT RECORDS
# ================================================================

empty_text_count = (
    df["clean_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Records with empty text:",
    empty_text_count
)

if empty_text_count == len(df):
    print(
        "\nWARNING:"
        "\nThere are no usable text fields."
        "\nThe model will rely only on numeric features."
    )


# ================================================================
# CELL 13 — TF-IDF VECTORIZATION
# ================================================================

use_text_features = (
    df["clean_text"]
    .str.strip()
    .ne("")
    .any()
)

if use_text_features:

    print("Creating TF-IDF features...")

    tfidf = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )

    X_text = tfidf.fit_transform(
        df["clean_text"]
    )

    print(
        "TF-IDF matrix shape:",
        X_text.shape
    )

else:

    X_text = None
    tfidf = None

    print("TF-IDF skipped because no usable text exists.")


# ================================================================
# CELL 14 — DISPLAY TF-IDF VOCABULARY
# ================================================================

if tfidf is not None:

    vocabulary = (
        tfidf
        .get_feature_names_out()
    )

    print(
        "Number of vocabulary terms:",
        len(vocabulary)
    )

    print("\nFirst 50 terms:")

    print(
        vocabulary[:50]
    )


# ================================================================
# CELL 15 — REDUCE NLP DIMENSIONS WITH SVD
# ================================================================

if X_text is not None:

    if X_text.shape[1] >= 3:

        actual_components = min(
            SVD_COMPONENTS,
            X_text.shape[1] - 1
        )

        svd = TruncatedSVD(
            n_components=actual_components,
            random_state=RANDOM_STATE
        )

        X_text_reduced = svd.fit_transform(
            X_text
        )

        explained_variance = (
            svd.explained_variance_ratio_.sum()
        )

        print(
            "Reduced NLP matrix:",
            X_text_reduced.shape
        )

        print(
            "Explained variance:",
            round(explained_variance, 4)
        )

    else:

        X_text_reduced = X_text.toarray()

        print(
            "Too few TF-IDF features for SVD."
        )

else:

    X_text_reduced = np.empty(
        (len(df), 0)
    )


# ================================================================
# CELL 16 — SCALE NUMERIC FEATURES
# ================================================================

if len(numeric_columns) > 0:

    scaler = StandardScaler()

    X_numeric = scaler.fit_transform(
        numeric_data
    )

    print(
        "Scaled numeric matrix:",
        X_numeric.shape
    )

else:

    X_numeric = np.empty(
        (len(df), 0)
    )


# ================================================================
# CELL 17 — COMBINE NLP + NUMERIC FEATURES
# ================================================================

feature_blocks = []

if X_text_reduced.shape[1] > 0:

    feature_blocks.append(
        X_text_reduced
    )

if X_numeric.shape[1] > 0:

    feature_blocks.append(
        X_numeric
    )

if not feature_blocks:

    raise ValueError(
        "No usable features were found."
    )

X = np.hstack(
    feature_blocks
)

print(
    "FINAL FEATURE MATRIX"
)

print("=" * 70)

print(
    "Rows:",
    X.shape[0]
)

print(
    "Features:",
    X.shape[1]
)


# ================================================================
# CELL 18 — CHECK FOR INVALID VALUES
# ================================================================

if np.isnan(X).any():

    print(
        "WARNING: NaN values detected."
    )

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

else:

    print(
        "No NaN values detected."
    )


# ================================================================
# CELL 19 — CHECK DATASET SIZE
# ================================================================

n_samples = X.shape[0]

if n_samples < 3:

    raise ValueError(
        "The dataset needs at least 3 records "
        "for K-Means clustering."
    )

max_possible_k = min(
    MAX_K,
    n_samples - 1
)

k_values = range(
    MIN_K,
    max_possible_k + 1
)

print(
    "K values to test:",
    list(k_values)
)


# ================================================================
# CELL 20 — TRAIN K-MEANS FOR MULTIPLE K VALUES
# ================================================================

inertias = []
silhouette_scores = []
calinski_scores = []
davies_scores = []

models = {}

for k in k_values:

    print(
        f"Training K-Means with K={k}..."
    )

    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=500,
        random_state=RANDOM_STATE
    )

    labels = model.fit_predict(X)

    models[k] = model

    inertias.append(
        model.inertia_
    )

    silhouette_scores.append(
        silhouette_score(
            X,
            labels
        )
    )

    calinski_scores.append(
        calinski_harabasz_score(
            X,
            labels
        )
    )

    davies_scores.append(
        davies_bouldin_score(
            X,
            labels
        )


# ================================================================
# CELL 21 — MODEL EVALUATION TABLE
# ================================================================

evaluation = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette": silhouette_scores,
    "Calinski_Harabasz": calinski_scores,
    "Davies_Bouldin": davies_scores
})

print(
    "K-MEANS EVALUATION"
)

display(evaluation)


# ================================================================
# CELL 22 — ELBOW PLOT
# ================================================================

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Inertia"],
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "K-Means Elbow Method"
)

plt.xticks(
    evaluation["K"]
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ================================================================
# CELL 23 — SILHOUETTE SCORE PLOT
# ================================================================

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Silhouette"],
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score by Number of Clusters"
)

plt.xticks(
    evaluation["K"]
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ================================================================
# CELL 24 — CALINSKI-HARABASZ PLOT
# ================================================================

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Calinski_Harabasz"],
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Calinski-Harabasz Score"
)

plt.title(
    "Calinski-Harabasz Score"
)

plt.xticks(
    evaluation["K"]
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ================================================================
# CELL 25 — DAVIES-BOULDIN PLOT
# ================================================================

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Davies_Bouldin"],
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Davies-Bouldin Score"
)

plt.title(
    "Davies-Bouldin Score"
)

plt.xticks(
    evaluation["K"]
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ================================================================
# CELL 26 — SELECT BEST K
# ================================================================

# Higher silhouette score is better.

best_k = int(
    evaluation.loc[
        evaluation["Silhouette"].idxmax(),
        "K"
    ]
)

best_silhouette = evaluation.loc[
    evaluation["Silhouette"].idxmax(),
    "Silhouette"
]

print(
    f"Recommended K: {best_k}"
)

print(
    f"Best silhouette score: "
    f"{best_silhouette:.4f}"
)


# ================================================================
# CELL 27 — TRAIN FINAL K-MEANS MODEL
# ================================================================

final_kmeans = KMeans(
    n_clusters=best_k,
    init="k-means++",
    n_init=50,
    max_iter=500,
    random_state=RANDOM_STATE
)

final_labels = final_kmeans.fit_predict(
    X
)

df["cluster"] = final_labels

print(
    "Final K-Means model trained."
)


# ================================================================
# CELL 28 — CLUSTER DISTRIBUTION
# ================================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

cluster_distribution = pd.DataFrame({
    "cluster": cluster_counts.index,
    "records": cluster_counts.values
})

cluster_distribution["percentage"] = (
    cluster_distribution["records"]
    / len(df)
    * 100
)

display(
    cluster_distribution
)


# ================================================================
# CELL 29 — CLUSTER DISTRIBUTION BAR CHART
# ================================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    data=cluster_distribution,
    x="cluster",
    y="records"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "Records per K-Means Cluster"
)

plt.show()


# ================================================================
# CELL 30 — 2D CLUSTER VISUALIZATION
# ================================================================

# Reduce the final combined feature matrix to 2 dimensions.

if X.shape[1] >= 3:

    visualization_svd = TruncatedSVD(
        n_components=2,
        random_state=RANDOM_STATE
    )

    X_2d = visualization_svd.fit_transform(
        X
    )

else:

    X_2d = X


plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    cmap="tab10",
    alpha=0.75,
    s=50
)

plt.xlabel(
    "Component 1"
)

plt.ylabel(
    "Component 2"
)

plt.title(
    f"K-Means Clusters — K={best_k}"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(
    True,
    alpha=0.2
)

plt.show()


# ================================================================
# CELL 31 — CALCULATE FINAL CLUSTER METRICS
# ================================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

final_calinski = calinski_harabasz_score(
    X,
    df["cluster"]
)

final_davies = davies_bouldin_score(
    X,
    df["cluster"]
)

print("=" * 70)
print("FINAL MODEL METRICS")
print("=" * 70)

print(
    f"Number of clusters: {best_k}"
)

print(
    f"Silhouette Score: {final_silhouette:.4f}"
)

print(
    f"Calinski-Harabasz Score: {final_calinski:.4f}"
)

print(
    f"Davies-Bouldin Score: {final_davies:.4f}"
)


# ================================================================
# CELL 32 — DISPLAY RECORDS FROM EACH CLUSTER
# ================================================================

display_columns = [
    column
    for column in df.columns
    if column not in [
        "combined_text",
        "clean_text"
    ]
]

for cluster_id in sorted(
    df["cluster"].unique()
):

    print("\n")
    print("=" * 80)
    print(
        f"CLUSTER {cluster_id}"
    )
    print("=" * 80)

    cluster_data = df[
        df["cluster"] == cluster_id
    ]

    print(
        "Number of records:",
        len(cluster_data)
    )

    display(
        cluster_data[
            display_columns
        ].head(10)
    )


# ================================================================
# CELL 33 — FIND TOP NLP TERMS FOR EACH CLUSTER
# ================================================================

cluster_terms = []

if tfidf is not None:

    terms = np.array(
        tfidf.get_feature_names_out()
    )

    for cluster_id in sorted(
        df["cluster"].unique()
    ):

        indexes = np.where(
            df["cluster"].values
            == cluster_id
        )[0]

        cluster_matrix = X_text[
            indexes
        ]

        mean_tfidf = np.asarray(
            cluster_matrix.mean(
                axis=0
            )
        ).flatten()

        top_indices = (
            mean_tfidf
            .argsort()[-TOP_TERMS:][::-1]
        )

        for rank, index in enumerate(
            top_indices,
            start=1
        ):

            cluster_terms.append({
                "cluster": cluster_id,
                "rank": rank,
                "term": terms[index],
                "tfidf_score": mean_tfidf[index]
            })


cluster_terms_df = pd.DataFrame(
    cluster_terms
)

display(
    cluster_terms_df
)


# ================================================================
# CELL 34 — PRINT TOP TERMS BY CLUSTER
# ================================================================

if not cluster_terms_df.empty:

    for cluster_id in sorted(
        cluster_terms_df["cluster"].unique()
    ):

        cluster_terms_subset = (
            cluster_terms_df[
                cluster_terms_df["cluster"]
                == cluster_id
            ]
            .sort_values("rank")
        )

        print(
            f"\nCluster {cluster_id} — "
            "Top NLP Terms"
        )

        print("-" * 70)

        for _, row in (
            cluster_terms_subset.iterrows()
        ):

            print(
                f"{int(row['rank']):2d}. "
                f"{row['term']} "
                f"({row['tfidf_score']:.4f})"
            )


# ================================================================
# CELL 35 — CREATE CLUSTER SUMMARY
# ================================================================

summary_rows = []

for cluster_id in sorted(
    df["cluster"].unique()
):

    cluster_data = df[
        df["cluster"] == cluster_id
    ]

    summary_rows.append({
        "cluster": cluster_id,
        "records": len(cluster_data),
        "percentage": (
            len(cluster_data)
            / len(df)
            * 100
        )
    })


cluster_summary = pd.DataFrame(
    summary_rows
)

display(
    cluster_summary
)


# ================================================================
# CELL 36 — NUMERIC CLUSTER PROFILE
# ================================================================

if numeric_columns:

    numeric_cluster_profile = (
        df.groupby("cluster")[
            numeric_columns
        ]
        .mean()
        .round(3)
    )

    print(
        "Average numeric values by cluster:"
    )

    display(
        numeric_cluster_profile
    )

else:

    numeric_cluster_profile = pd.DataFrame()

    print(
        "No numeric columns available."
    )


# ================================================================
# CELL 37 — HEATMAP OF NUMERIC CLUSTER PROFILES
# ================================================================

if not numeric_cluster_profile.empty:

    plt.figure(
        figsize=(
            max(10, len(numeric_columns) * 1.2),
            6
        )
    )

    sns.heatmap(
        numeric_cluster_profile,
        annot=True,
        fmt=".2f",
        cmap="viridis"
    )

    plt.title(
        "Average Numeric Features by Cluster"
    )

    plt.xlabel(
        "Numeric Feature"
    )

    plt.ylabel(
        "Cluster"
    )

    plt.tight_layout()

    plt.show()


# ================================================================
# CELL 38 — CLUSTER CENTERS
# ================================================================

cluster_centers = pd.DataFrame(
    final_kmeans.cluster_centers_
)

cluster_centers.index.name = "cluster"

print(
    "K-Means cluster centers:"
)

display(
    cluster_centers.head()
)


# ================================================================
# CELL 39 — FIND NEAREST RECORDS TO EACH CENTROID
# ================================================================

# Distance from every record to its assigned centroid

distances = final_kmeans.transform(
    X
)

df["distance_to_centroid"] = (
    distances[
        np.arange(len(df)),
        df["cluster"].values
    ]
)

print(
    "Records closest to each centroid:"
)

for cluster_id in sorted(
    df["cluster"].unique()
):

    nearest = (
        df[
            df["cluster"] == cluster_id
        ]
        .sort_values(
            "distance_to_centroid"
        )
        .head(5)
    )

    print(
        f"\nCluster {cluster_id}"
    )

    display(
        nearest[
            display_columns
            + ["distance_to_centroid"]
        ]
    )


# ================================================================
# CELL 40 — REMOVE HELPER COLUMNS IF DESIRED
# ================================================================

# Keep cluster and centroid distance,
# but remove temporary NLP columns.

df_output = df.drop(
    columns=[
        "combined_text",
        "clean_text"
    ],
    errors="ignore"
)

print(
    "Output dataset shape:",
    df_output.shape
)

display(
    df_output.head()
)


# ================================================================
# CELL 41 — SAVE CLUSTERED DATASET
# ================================================================

df_output.to_csv(
    OUTPUT_CLUSTERED,
    index=False
)

print(
    f"Saved clustered dataset:\n"
    f"{OUTPUT_CLUSTERED}"
)


# ================================================================
# CELL 42 — SAVE CLUSTER SUMMARY
# ================================================================

cluster_summary.to_csv(
    OUTPUT_SUMMARY,
    index=False
)

print(
    f"Saved cluster summary:\n"
    f"{OUTPUT_SUMMARY}"
)


# ================================================================
# CELL 43 — SAVE IMPORTANT NLP TERMS
# ================================================================

if not cluster_terms_df.empty:

    cluster_terms_df.to_csv(
        OUTPUT_TERMS,
        index=False
    )

    print(
        f"Saved cluster NLP terms:\n"
        f"{OUTPUT_TERMS}"
    )

else:

    print(
        "No NLP terms were available to save."
    )


# ================================================================
# CELL 44 — FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("FINAL K-MEANS + NLP REPORT")
print("=" * 80)

print(
    f"Dataset: {FILE_PATH}"
)

print(
    f"Original rows: {original_rows}"
)

print(
    f"Rows after duplicate removal: {len(df)}"
)

print(
    f"Text columns: {len(text_columns)}"
)

print(
    f"Numeric columns: {len(numeric_columns)}"
)

if tfidf is not None:

    print(
        f"TF-IDF vocabulary: {X_text.shape[1]}"
    )

else:

    print(
        "TF-IDF vocabulary: 0"
    )

print(
    f"Final feature count: {X.shape[1]}"
)

print(
    f"Selected K: {best_k}"
)

print(
    f"Final Silhouette Score: "
    f"{final_silhouette:.4f}"
)

print(
    f"Final Calinski-Harabasz Score: "
    f"{final_calinski:.4f}"
)

print(
    f"Final Davies-Bouldin Score: "
    f"{final_davies:.4f}"
)

print("\nCluster sizes:")

display(
    cluster_distribution
)

print("\nOutput files:")

print(
    f"1. {OUTPUT_CLUSTERED}"
)

print(
    f"2. {OUTPUT_SUMMARY}"
)

print(
    f"3. {OUTPUT_TERMS}"
)

print("\nProcessing complete.")

SyntaxError: '(' was never closed (2907346743.py, line 603)